# LeetCode 1852 — Distinct Numbers in Each Subarray bằng Python

Link đề: https://leetcode.com/problems/distinct-numbers-in-each-subarray/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Sliding Window

> **Ghi chú (bài Premium 🔒):** đề trong notebook này được viết lại theo hiểu biết về bài và theo bản mô tả công khai của kho doocs/leetcode. Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc. (Bản mô tả viết "với mỗi `0 ≤ i < n − k`" nhưng các ví dụ cho `n − k + 1` đáp án — notebook theo các ví dụ: có đúng `n − k + 1` cửa sổ.)

Skeleton LeetCode:

```python
class Solution:
    def distinctNumbers(self, nums: List[int], k: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Cài **cửa sổ trượt kích thước cố định**: thêm phần tử bên phải, bỏ phần tử bên trái, cập nhật đáp án trong `O(1)`.
2. Duy trì **số phần tử khác nhau** bằng bảng đếm: tăng khi một giá trị xuất hiện lần đầu trong cửa sổ, giảm khi số đếm của nó về 0.
3. So sánh `Counter` (xóa khóa khi về 0, dùng `len`) với **mảng đếm** + biến `distinct` (nhanh hơn khi giá trị nhỏ).
4. Một cách nhìn khác: mỗi vị trí `j` "đóng góp" cho những cửa sổ mà `nums[j]` là **lần xuất hiện đầu tiên** trong cửa sổ ⇒ **mảng hiệu** (difference array) với vị trí xuất hiện trước `prev[j]`.
5. Đối chiếu với vét cạn `len(set(window))` trên dữ liệu ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n` và số nguyên `k`. Với **mỗi mảng con độ dài `k`** `nums[i..i+k−1]` (`0 ≤ i ≤ n − k`), đếm số phần tử **khác nhau**. Trả về mảng `ans` gồm `n − k + 1` kết quả theo thứ tự `i`.

**Ví dụ 1:** `nums = [1,2,3,2,2,1,3], k = 3` → `[3,2,2,2,3]`.
- `[1,2,3]` → 3; `[2,3,2]` → 2; `[3,2,2]` → 2; `[2,2,1]` → 2; `[2,1,3]` → 3.

**Ví dụ 2:** `nums = [1,1,1,1,2,3,4], k = 4` → `[1,2,3,4]`.

**Ràng buộc:** `1 ≤ k ≤ n ≤ 10⁵`; `1 ≤ nums[i] ≤ 10⁵`.

In [1]:
import random
import time
from collections import Counter, defaultdict
from typing import List

examples = [([1, 2, 3, 2, 2, 1, 3], 3, [3, 2, 2, 2, 3]), ([1, 1, 1, 1, 2, 3, 4], 4, [1, 2, 3, 4])]
for nums, k, exp in examples:
    print(f"nums={nums}, k={k} -> kỳ vọng {exp}")

nums=[1, 2, 3, 2, 2, 1, 3], k=3 -> kỳ vọng [3, 2, 2, 2, 3]
nums=[1, 1, 1, 1, 2, 3, 4], k=4 -> kỳ vọng [1, 2, 3, 4]


## 3. Kiến thức nền

### 3.1 `set` và `len` — số phần tử khác nhau

In [2]:
print(len(set([2, 3, 2])), len(set([1, 1, 1, 1])), len({1, 2, 3}))

2 1 3


### 3.2 Cửa sổ trượt kích thước cố định

Cửa sổ `[i, i+k−1]` chuyển sang `[i+1, i+k]`: **bỏ** `nums[i]`, **thêm** `nums[i+k]`. Thay vì tính lại cả cửa sổ (`O(k)`), chỉ cập nhật hai phần tử (`O(1)`).

In [3]:
nums, k = [1, 2, 3, 2, 2], 3
for i in range(len(nums) - k + 1):
    print(f"cửa sổ i={i}: {nums[i:i+k]}" + (f" | bỏ {nums[i-1]}, thêm {nums[i+k-1]}" if i else ""))

cửa sổ i=0: [1, 2, 3]
cửa sổ i=1: [2, 3, 2] | bỏ 1, thêm 2
cửa sổ i=2: [3, 2, 2] | bỏ 2, thêm 2


### 3.3 `Counter`: đếm, giảm và xóa khóa về 0

`len(cnt)` đếm số **khóa**; khóa có giá trị 0 vẫn được tính nếu không xóa ⇒ phải `del cnt[x]` (hoặc `pop`) khi về 0.

In [4]:
cnt = Counter([2, 3, 2]); print(cnt, len(cnt))
cnt[3] -= 1; print("sau khi giảm 3 về 0:", cnt, "| len =", len(cnt), "(vẫn tính khóa 3!)")
del cnt[3]; print("sau khi xóa:", cnt, "| len =", len(cnt))

Counter({2: 2, 3: 1}) 2
sau khi giảm 3 về 0: Counter({2: 2, 3: 0}) | len = 2 (vẫn tính khóa 3!)
sau khi xóa: Counter({2: 2}) | len = 1


### 3.4 Mảng hiệu (difference array)

Muốn cộng `+1` cho mọi chỉ số trong đoạn `[l, r]` của một mảng `ans`: `d[l] += 1`, `d[r+1] −= 1`; cuối cùng `ans = tổng tiền tố của d`. Nhiều đoạn chỉ tốn `O(1)` mỗi đoạn.

In [5]:
from itertools import accumulate
d = [0] * 6
for l, r in [(0, 2), (1, 4), (3, 3)]:
    d[l] += 1; d[r + 1] -= 1
print("d =", d, "-> ans =", list(accumulate(d))[:5])

d = [1, 1, 0, 0, -1, -1] -> ans = [1, 2, 2, 2, 1]


## 4. Ý tưởng nền tảng / trực giác

Theo dõi **Ví dụ 1** với bảng đếm khi cửa sổ trượt (`k = 3`):

| i | cửa sổ | bỏ | thêm | bảng đếm | số khác nhau |
|---|---|---|---|---|---|
| 0 | [1,2,3] | — | — | {1:1, 2:1, 3:1} | 3 |
| 1 | [2,3,2] | 1 (về 0 ⇒ −1) | 2 (đã có) | {2:2, 3:1} | 2 |
| 2 | [3,2,2] | 2 (còn 1) | 2 | {2:2, 3:1} | 2 |
| 3 | [2,2,1] | 3 (về 0 ⇒ −1) | 1 (mới ⇒ +1) | {2:2, 1:1} | 2 |
| 4 | [2,1,3] | 2 (còn 1) | 3 (mới ⇒ +1) | {2:1, 1:1, 3:1} | 3 |

Số phần tử khác nhau chỉ thay đổi khi: thêm một giá trị **chưa có** (+1), hoặc bỏ **bản cuối cùng** của một giá trị (−1).

## 5. Cách 1 — Brute Force / baseline: `len(set(window))` cho mỗi cửa sổ

In [6]:
def distinct_bruteforce(nums, k):
    return [len(set(nums[i:i + k])) for i in range(len(nums) - k + 1)]

for nums, k, exp in examples:
    got = distinct_bruteforce(nums, k); print(got, "kỳ vọng", exp); assert got == exp

[3, 2, 2, 2, 3] kỳ vọng [3, 2, 2, 2, 3]
[1, 2, 3, 4] kỳ vọng [1, 2, 3, 4]


**Độ phức tạp:** `(n − k + 1)` cửa sổ × `O(k)` ⇒ `O(n·k)`, tệ nhất khi `k ≈ n/2`: `2,5·10⁹` ⇒ **TLE**.

**Nút thắt:** hai cửa sổ kề nhau chung `k − 1` phần tử nhưng ta dựng `set` lại từ đầu.

In [7]:
arr = [random.Random(0).randint(1, 1000) for _ in range(20000)]
t = time.perf_counter(); distinct_bruteforce(arr, 10000); print(f"vét cạn n=20000, k=10000: {(time.perf_counter()-t)*1000:.0f} ms")

vét cạn n=20000, k=10000: 967 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (cửa sổ trượt + bảng đếm):** giữ `cnt[x]` = số lần `x` xuất hiện trong cửa sổ và `distinct` = số `x` có `cnt[x] > 0`.
- Thêm `x`: **nếu** `cnt[x] == 0` trước khi tăng **thì** `distinct += 1`; `cnt[x] += 1`.
- Bỏ `y`: `cnt[y] −= 1`; **nếu** `cnt[y] == 0` **thì** `distinct −= 1`.

**Invariant:** sau khi xử lý cửa sổ bắt đầu tại `i`, `cnt` mô tả đúng `nums[i..i+k−1]` và `distinct = |{x : cnt[x] > 0}|`.

**Góc nhìn đóng góp:** gọi `prev[j]` = vị trí xuất hiện **gần nhất trước `j`** của `nums[j]` (hoặc `−1`). Vị trí `j` là "lần đầu" của `nums[j]` trong cửa sổ `[i, i+k−1]` ⇔ `prev[j] < i ≤ j` và `i ≥ j − k + 1`. Vậy `j` cộng 1 cho mọi `i ∈ [max(prev[j] + 1, j − k + 1), min(j, n − k)]` ⇒ mảng hiệu.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cửa sổ trượt với `Counter` (xóa khóa khi về 0)

In [8]:
def distinct_counter(nums, k):
    cnt = Counter(nums[:k]); ans = [len(cnt)]
    for i in range(k, len(nums)):
        cnt[nums[i]] += 1
        out = nums[i - k]
        cnt[out] -= 1
        if cnt[out] == 0:
            del cnt[out]
        ans.append(len(cnt))
    return ans

print([distinct_counter(n, k) for n, k, _ in examples])

[[3, 2, 2, 2, 3], [1, 2, 3, 4]]


**Code nộp LeetCode (Cách 2):** đổi tên thành `distinctNumbers`. Thứ tự thêm trước rồi bỏ sau an toàn: nếu `nums[i] == nums[i−k]`, số đếm tăng rồi giảm về như cũ, không xóa nhầm. **Độ phức tạp:** Time `O(n)`, Space `O(min(k, V))`.

### 7.2 Cách 3 — Mảng đếm + biến `distinct`

In [9]:
class Solution2:
    def distinctNumbers(self, nums: List[int], k: int) -> List[int]:
        cnt = [0] * (max(nums) + 1)
        distinct = 0
        ans = []
        for i, x in enumerate(nums):
            if cnt[x] == 0:
                distinct += 1
            cnt[x] += 1
            if i >= k:
                y = nums[i - k]
                cnt[y] -= 1
                if cnt[y] == 0:
                    distinct -= 1
            if i >= k - 1:
                ans.append(distinct)
        return ans

print([Solution2().distinctNumbers(n, k) for n, k, _ in examples])

[[3, 2, 2, 2, 3], [1, 2, 3, 4]]


**Độ phức tạp Cách 3:** Time `O(n + V)`, Space `O(V)` với `V = max(nums) ≤ 10⁵`. Thường nhanh nhất trong Python vì tránh băm.

### 7.3 Cách 4 — Đóng góp theo "lần xuất hiện trước" + mảng hiệu

In [10]:
def distinct_contribution(nums, k):
    n = len(nums); m = n - k + 1
    d = [0] * (m + 1); last = {}
    for j, x in enumerate(nums):
        p = last.get(x, -1); last[x] = j
        lo = max(p + 1, j - k + 1); hi = min(j, m - 1)
        if lo <= hi:
            d[lo] += 1; d[hi + 1] -= 1
    return list(accumulate(d))[:m]

print([distinct_contribution(n, k) for n, k, _ in examples])

[[3, 2, 2, 2, 3], [1, 2, 3, 4]]


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)`. Ưu điểm: tách được "mỗi phần tử đóng góp cho đoạn cửa sổ nào" — kỹ thuật dùng lại cho các bài đếm "số phần tử khác nhau" trên **mọi** mảng con (ví dụ 2262 Total Appeal).

### 7.4 Cách 5 — Cửa sổ trượt với `dict` thường (`get`)

Giống Cách 2 nhưng dùng `dict` thuần và `distinct` riêng, không phụ thuộc `len(dict)` — phong cách dễ chuyển sang C++/Java (`unordered_map`).

In [11]:
def distinct_dict(nums, k):
    cnt = {}; distinct = 0; ans = []
    for i, x in enumerate(nums):
        c = cnt.get(x, 0)
        if c == 0: distinct += 1
        cnt[x] = c + 1
        if i >= k:
            y = nums[i - k]; cnt[y] -= 1
            if cnt[y] == 0: distinct -= 1
        if i >= k - 1: ans.append(distinct)
    return ans

print([distinct_dict(n, k) for n, k, _ in examples])

[[3, 2, 2, 2, 3], [1, 2, 3, 4]]


## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 2** (`[1,1,1,1,2,3,4]`, `k = 4`):

In [12]:
def trace_array(nums, k):
    cnt = defaultdict(int); distinct = 0; ans = []
    for i, x in enumerate(nums):
        note = []
        if cnt[x] == 0: distinct += 1; note.append(f"thêm {x} (mới, +1)")
        else: note.append(f"thêm {x} (đã có)")
        cnt[x] += 1
        if i >= k:
            y = nums[i - k]; cnt[y] -= 1
            note.append(f"bỏ {y}" + (" (về 0, −1)" if cnt[y] == 0 else f" (còn {cnt[y]})"))
            if cnt[y] == 0: distinct -= 1
        rec = ""
        if i >= k - 1:
            ans.append(distinct); rec = f" -> ans[{i - k + 1}] = {distinct}"
        print(f"i={i}: {', '.join(note):<32} cửa sổ={nums[max(0, i - k + 1):i + 1]}{rec}")
    return ans

assert trace_array([1, 1, 1, 1, 2, 3, 4], 4) == [1, 2, 3, 4]

i=0: thêm 1 (mới, +1)                 cửa sổ=[1]
i=1: thêm 1 (đã có)                   cửa sổ=[1, 1]
i=2: thêm 1 (đã có)                   cửa sổ=[1, 1, 1]
i=3: thêm 1 (đã có)                   cửa sổ=[1, 1, 1, 1] -> ans[0] = 1
i=4: thêm 2 (mới, +1), bỏ 1 (còn 3)   cửa sổ=[1, 1, 1, 2] -> ans[1] = 2
i=5: thêm 3 (mới, +1), bỏ 1 (còn 2)   cửa sổ=[1, 1, 2, 3] -> ans[2] = 3
i=6: thêm 4 (mới, +1), bỏ 1 (còn 1)   cửa sổ=[1, 2, 3, 4] -> ans[3] = 4


Trace Cách 4 (đóng góp) trên Ví dụ 1: mỗi `j` cộng 1 cho đoạn cửa sổ `[lo, hi]`.

In [13]:
nums, k = [1, 2, 3, 2, 2, 1, 3], 3; n = len(nums); m = n - k + 1; last = {}
for j, x in enumerate(nums):
    p = last.get(x, -1); last[x] = j
    lo, hi = max(p + 1, j - k + 1), min(j, m - 1)
    print(f"j={j} (x={x}, prev={p}): cộng cho cửa sổ {list(range(lo, hi + 1)) if lo <= hi else '∅'}")
print("kết quả:", distinct_contribution(nums, k))

j=0 (x=1, prev=-1): cộng cho cửa sổ [0]
j=1 (x=2, prev=-1): cộng cho cửa sổ [0, 1]
j=2 (x=3, prev=-1): cộng cho cửa sổ [0, 1, 2]
j=3 (x=2, prev=1): cộng cho cửa sổ [2, 3]
j=4 (x=2, prev=3): cộng cho cửa sổ [4]
j=5 (x=1, prev=0): cộng cho cửa sổ [3, 4]
j=6 (x=3, prev=2): cộng cho cửa sổ [4]
kết quả: [3, 2, 2, 2, 3]


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — cửa sổ trượt với `Counter`.

In [14]:
class Solution:
    def distinctNumbers(self, nums: List[int], k: int) -> List[int]:
        cnt = Counter(nums[:k])
        ans = [len(cnt)]
        for i in range(k, len(nums)):
            cnt[nums[i]] += 1
            out = nums[i - k]
            cnt[out] -= 1
            if cnt[out] == 0:
                del cnt[out]
            ans.append(len(cnt))
        return ans

for nums, k, exp in examples:
    got = Solution().distinctNumbers(nums, k); print(got, exp); assert got == exp

[3, 2, 2, 2, 3] [3, 2, 2, 2, 3]
[1, 2, 3, 4] [1, 2, 3, 4]


## 10. Vì sao thuật toán đúng

**Invariant (Cách 2):** trước khi ghi `ans` cho cửa sổ bắt đầu tại `s`, `cnt` = `Counter(nums[s..s+k−1])` và **không** chứa khóa có giá trị 0.
- Khởi tạo: `Counter(nums[:k])` ✓.
- Chuyển cửa sổ `s → s+1`: tăng `cnt[nums[s+k]]`, giảm `cnt[nums[s]]` ⇒ đúng là đếm của `nums[s+1..s+k]`; nếu một khóa về 0 thì xóa ⇒ giữ điều kiện "không khóa 0". Khi `nums[s+k] == nums[s]`, tăng rồi giảm trả về giá trị cũ `≥ 1` ⇒ không xóa nhầm.
- Vì không có khóa 0, `len(cnt)` = số giá trị có mặt = số phần tử khác nhau. ∎

**Cách 4:** trong cửa sổ `[i, i+k−1]`, mỗi giá trị khác nhau có **đúng một** lần xuất hiện đầu tiên `j` (thỏa `prev[j] < i ≤ j`). Đếm các `j` như vậy = số phần tử khác nhau; mảng hiệu cộng đúng các đóng góp đó.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) ≤ 10⁵`, `V = max(nums) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. `set` mỗi cửa sổ | `O(n · k)` | `O(k)` |
| 2. Counter trượt | `O(n)` | `O(min(k, V))` |
| 3. Mảng đếm | `O(n + V)` | `O(V)` |
| 4. Đóng góp + mảng hiệu | `O(n)` | `O(n)` |
| 5. dict + distinct | `O(n)` | `O(min(k, V))` |

In [15]:
rng = random.Random(1852)
big = [rng.randint(1, 10 ** 5) for _ in range(10 ** 5)]
for name, f in [("Counter", Solution().distinctNumbers), ("mảng đếm", Solution2().distinctNumbers),
                ("đóng góp", distinct_contribution), ("dict", distinct_dict)]:
    t = time.perf_counter(); r = f(big, 50000); print(f"{name:<9} {(time.perf_counter()-t)*1000:6.0f} ms | 3 đầu {r[:3]}")

Counter       27 ms | 3 đầu [39364, 39364, 39365]
mảng đếm      15 ms | 3 đầu [39364, 39364, 39365]
đóng góp      29 ms | 3 đầu [39364, 39364, 39365]
dict          26 ms | 3 đầu [39364, 39364, 39365]


## 12. Bộ test

In [16]:
ALL = {"counter": distinct_counter, "array": Solution2().distinctNumbers, "contrib": distinct_contribution,
       "dict": distinct_dict, "submit": Solution().distinctNumbers}
def run_all(nums, k):
    ref = distinct_bruteforce(nums, k)
    for name, f in ALL.items():
        assert f(list(nums), k) == ref, (name, nums, k)
    assert len(ref) == len(nums) - k + 1
    return ref

for nums, k, exp in examples:
    assert run_all(nums, k) == exp
edge = [([5], 1, [1]), ([1, 2, 3], 1, [1, 1, 1]), ([1, 2, 3], 3, [3]), ([7, 7, 7, 7], 2, [1, 1, 1]), ([1, 2, 1, 2], 2, [2, 2, 2]),
        ([1, 2, 1, 2], 4, [2]), ([100000, 1, 100000], 2, [2, 2])]
for nums, k, exp in edge:
    assert run_all(nums, k) == exp, (nums, k)
rng = random.Random(18520)
for _ in range(800):
    n = rng.randint(1, 40); nums = [rng.randint(1, rng.choice([2, 5, 20, 10 ** 5])) for _ in range(n)]
    run_all(nums, rng.randint(1, n))
print("Ví dụ, edge case, 800 test ngẫu nhiên: OK")

Ví dụ, edge case, 800 test ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

1. Hai cửa sổ kề nhau chênh nhau tối đa 1: `|ans[i+1] − ans[i]| ≤ 1` (chỉ một phần tử vào, một phần tử ra).
2. `1 ≤ ans[i] ≤ min(k, số giá trị khác nhau của cả mảng)`.
3. Nếu `nums` toàn phần tử khác nhau thì mọi `ans[i] = k`.

In [17]:
rng = random.Random(21)
for _ in range(400):
    n = rng.randint(1, 50); nums = [rng.randint(1, 8) for _ in range(n)]; k = rng.randint(1, n)
    a = Solution().distinctNumbers(nums, k)
    assert all(abs(a[i + 1] - a[i]) <= 1 for i in range(len(a) - 1))
    assert all(1 <= v <= min(k, len(set(nums))) for v in a)
    perm = rng.sample(range(1, 100), n)
    assert Solution().distinctNumbers(perm, k) == [k] * (n - k + 1)
print("3 tính chất đúng trên 400 input")

3 tính chất đúng trên 400 input


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không xóa khóa về 0 trong `Counter` nhưng vẫn dùng `len(cnt)`

In [18]:
def wrong_no_delete(nums, k):
    cnt = Counter(nums[:k]); ans = [len(cnt)]
    for i in range(k, len(nums)):
        cnt[nums[i]] += 1; cnt[nums[i - k]] -= 1          # SAI: khóa 0 vẫn được len() đếm
        ans.append(len(cnt))
    return ans

nums, k, exp = examples[0]
print("sai :", wrong_no_delete(nums, k), "| đúng:", exp)

sai : [3, 3, 3, 3, 3] | đúng: [3, 2, 2, 2, 3]


**Vì sao sai:** `len(Counter)` đếm **khóa**, kể cả khóa giá trị 0. **Sửa:** `del cnt[x]` khi về 0 (hoặc đếm `distinct` riêng).

### Lỗi 2 — Sai số lượng cửa sổ (`n − k` thay vì `n − k + 1`)

In [19]:
def wrong_count_windows(nums, k):
    return [len(set(nums[i:i + k])) for i in range(len(nums) - k)]       # SAI: thiếu cửa sổ cuối

nums, k, exp = examples[1]
print("sai :", wrong_count_windows(nums, k), "| đúng:", exp)

sai : [1, 2, 3] | đúng: [1, 2, 3, 4]


**Vì sao sai:** có `n − k + 1` vị trí bắt đầu (`0..n−k`). **Sửa:** `range(n − k + 1)`.

### Lỗi 3 — Bỏ phần tử trái **trước** khi ghi đáp án của cửa sổ đầu / lệch chỉ số

In [20]:
def wrong_offset(nums, k):
    cnt = defaultdict(int); distinct = 0; ans = []
    for i, x in enumerate(nums):
        if cnt[x] == 0: distinct += 1
        cnt[x] += 1
        if i >= k - 1:
            y = nums[i - k + 1]                            # SAI: bỏ phần tử còn trong cửa sổ hiện tại
            cnt[y] -= 1
            if cnt[y] == 0: distinct -= 1
            ans.append(distinct)
    return ans

nums, k, exp = examples[0]
print("sai :", wrong_offset(nums, k), "| đúng:", exp)

sai : [2, 2, 1, 2, 2] | đúng: [3, 2, 2, 2, 3]


**Vì sao sai:** phần tử cần bỏ khi thêm `nums[i]` là `nums[i − k]` (vừa rời cửa sổ), và phải ghi đáp án **sau** khi cửa sổ đúng `k` phần tử. **Sửa:** `if i >= k: bỏ nums[i−k]`; `if i >= k−1: ghi`.

### Lỗi 4 — Dùng `set` và `remove` (mất thông tin số lần)

In [21]:
def wrong_set_slide(nums, k):
    window = set(nums[:k]); ans = [len(window)]
    for i in range(k, len(nums)):
        window.discard(nums[i - k])                         # SAI: xóa luôn dù còn bản khác trong cửa sổ
        window.add(nums[i])
        ans.append(len(window))
    return ans

nums, k, exp = examples[0]
print("sai :", wrong_set_slide(nums, k), "| đúng:", exp)

sai : [3, 2, 2, 2, 2] | đúng: [3, 2, 2, 2, 3]


**Vì sao sai:** `set` không biết `2` còn xuất hiện ở chỗ khác trong cửa sổ; xóa `2` khi bản bên trái rời đi làm mất giá trị. **Sửa:** đếm số lần (`Counter`/mảng đếm).

### Lỗi 5 — Bỏ trước rồi thêm, khi phần tử thêm và bỏ bằng nhau (với `Counter` + `del`)

Thứ tự "bỏ trước, thêm sau" **cũng đúng** nếu xóa khóa khi về 0 rồi thêm lại; nhưng nếu dùng điều kiện `distinct` riêng mà cập nhật sai thứ tự (kiểm tra `cnt[x] == 0` **sau** khi tăng) thì sai:

In [22]:
def wrong_check_after(nums, k):
    cnt = defaultdict(int); distinct = 0; ans = []
    for i, x in enumerate(nums):
        cnt[x] += 1
        if cnt[x] == 0: distinct += 1                       # SAI: kiểm tra sau khi tăng -> không bao giờ đúng
        if i >= k:
            y = nums[i - k]; cnt[y] -= 1
            if cnt[y] == 0: distinct -= 1
        if i >= k - 1: ans.append(distinct)
    return ans

nums, k, exp = examples[1]
print("sai :", wrong_check_after(nums, k), "| đúng:", exp)

sai : [0, 0, 0, 0] | đúng: [1, 2, 3, 4]


**Vì sao sai:** phải kiểm tra "trước đó chưa có" (`cnt[x] == 0` **trước** khi tăng), hoặc `cnt[x] == 1` **sau** khi tăng. **Sửa:** như Cách 3.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Với **mỗi** mảng con độ dài `k`…" ⇒ **cửa sổ trượt cố định**, cập nhật `O(1)` mỗi bước.
- "Số phần tử khác nhau" ⇒ bảng đếm + biến `distinct` (tăng khi 0→1, giảm khi 1→0).
- `len(Counter)` chỉ đúng nếu xóa khóa về 0.
- "Tổng/đếm trên mọi mảng con" ⇒ nghĩ tới **đóng góp của từng phần tử** (`prev[j]`) + mảng hiệu.

## 15. Biến thể của bài

1. **Số phần tử xuất hiện đúng một lần** trong mỗi cửa sổ: thêm biến `once` cập nhật khi đếm qua 1 (Bài tập 5).
2. **2461 Maximum Sum of Distinct Subarrays With Length K**: cửa sổ cố định + điều kiện `distinct == k`.
3. **Cửa sổ không cố định**: số mảng con có đúng `K` phần tử khác nhau (**992**) — "atMost(K) − atMost(K−1)".
4. **2262 Total Appeal of A String**: tổng số phần tử khác nhau của **mọi** chuỗi con — đóng góp `(j − prev[j]) · (n − j)`.
5. **Truy vấn đoạn bất kỳ `[l, r]`** (không cố định độ dài): offline + Fenwick theo `prev` (bài kinh điển "D-query").

**Minh họa biến thể 3 — đếm số mảng con (độ dài bất kỳ) có đúng `K` phần tử khác nhau (LeetCode 992).** Mẹo: `exactly(K) = atMost(K) − atMost(K − 1)`, trong đó `atMost(K)` đếm bằng cửa sổ **co giãn**: với mỗi `r`, thu `l` cho tới khi cửa sổ còn `≤ K` giá trị khác nhau; mọi mảng con kết thúc tại `r` bắt đầu từ `l..r` đều hợp lệ (`r − l + 1` mảng).

In [23]:
def at_most(nums, K):
    cnt = defaultdict(int); distinct = 0; l = 0; total = 0
    for r, x in enumerate(nums):
        if cnt[x] == 0: distinct += 1
        cnt[x] += 1
        while distinct > K:
            y = nums[l]; cnt[y] -= 1
            if cnt[y] == 0: distinct -= 1
            l += 1
        total += r - l + 1
    return total

def exactly_k_distinct(nums, K):
    return at_most(nums, K) - (at_most(nums, K - 1) if K > 0 else 0)

rng = random.Random(22)
for _ in range(300):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(1, 14))]; K = rng.randint(1, 4)
    brute = sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if len(set(nums[i:j])) == K)
    assert exactly_k_distinct(nums, K) == brute
print("992 trên [1,2,1,2,3], K=2:", exactly_k_distinct([1, 2, 1, 2, 3], 2), "(LeetCode cho 7)")

992 trên [1,2,1,2,3], K=2: 7 (LeetCode cho 7)


**Minh họa biến thể 4 — tổng số phần tử khác nhau của mọi mảng con (ý tưởng của 2262).** Vị trí `j` là "lần đầu" của `nums[j]` trong mảng con `[i..r]` ⇔ `prev[j] < i ≤ j ≤ r` ⇒ đóng góp `(j − prev[j]) · (n − j)` — đúng tư duy "đóng góp" của Cách 4.

In [24]:
def total_distinct_all_subarrays(nums):
    n = len(nums); last = {}; total = 0
    for j, x in enumerate(nums):
        p = last.get(x, -1); total += (j - p) * (n - j); last[x] = j
    return total

rng = random.Random(23)
for _ in range(300):
    nums = [rng.randint(1, 5) for _ in range(rng.randint(1, 15))]
    assert total_distinct_all_subarrays(nums) == sum(len(set(nums[i:j])) for i in range(len(nums)) for j in range(i + 1, len(nums) + 1))
print("tổng trên [1,2,1]:", total_distinct_all_subarrays([1, 2, 1]))

tổng trên [1,2,1]: 9


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `set` mỗi cửa sổ | Tính lại | `O(nk)` | `O(k)` | Kiểm chứng |
| 2. Counter trượt | Thêm/bỏ, xóa khóa 0 | `O(n)` | `O(k)` | **Bản nộp**, ngắn |
| 3. Mảng đếm | `distinct` riêng | `O(n + V)` | `O(V)` | Giá trị nhỏ, nhanh nhất |
| 4. Đóng góp | `prev[j]` + mảng hiệu | `O(n)` | `O(n)` | Tổng quát hóa cho mọi mảng con |
| 5. dict + distinct | Như 3, không cần V | `O(n)` | `O(k)` | Chuyển ngôn ngữ dễ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([1,2,3,4], 2)`; (b) `([4,4,4], 1)`; (c) `([1,2,1,3,1], 3)`; (d) `([2,2,3,3,2,2], 4)`.

**Bài 2 (tính tay):** Trace cửa sổ trượt (bảng đếm, `distinct`) cho `nums = [5,3,5,5,3,1]`, `k = 3`.

**Bài 3 (điền TODO):** Hoàn thiện `add(cnt, x)` và `remove(cnt, x)` trả **thay đổi** của `distinct` (+1, 0, −1).

**Bài 4 (tự viết lại):** Viết `max_distinct_window(nums, k)` trả `(số khác nhau lớn nhất, chỉ số cửa sổ đầu tiên đạt)`.

**Bài 5 (mini challenge):** Viết `unique_once(nums, k)`: với mỗi cửa sổ độ dài `k`, đếm số giá trị xuất hiện **đúng một lần** trong cửa sổ, `O(n)`.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def add_todo(cnt, x):
    # TODO: nếu cnt.get(x, 0) == 0 thì delta = 1, ngược lại 0; tăng cnt[x]; trả delta
    pass

def remove_todo(cnt, x):
    # TODO: giảm cnt[x]; nếu về 0 thì xóa và trả -1, ngược lại trả 0
    pass

In [26]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def max_distinct_window(nums, k):
    # TODO: dùng Solution().distinctNumbers rồi tìm max và chỉ số đầu tiên
    pass

def unique_once(nums, k):
    # TODO: biến once: khi cnt[x] đổi 0->1 thì once += 1, 1->2 thì once -= 1; bỏ: 1->0 once -= 1, 2->1 once += 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: liệt kê cửa sổ. Mức 2: đếm khác nhau. Mức 3: (a) `[2,2,2]`; (b) `[1,1,1]`; (c) `[2,3,2]`; (d) `[2,2,2]`.

**Bài 2.** Mức 1: cửa sổ đầu `[5,3,5]` ⇒ `{5:2, 3:1}` ⇒ 2. Mức 2: bỏ 5 (còn 1), thêm 5 ⇒ 2; bỏ 3 (về 0, −1), thêm 3 (+1) ⇒ 2. Mức 3: bỏ 5 (còn 1), thêm 1 (+1) ⇒ 3 ⇒ `[2, 2, 2, 3]`.

**Bài 3.** Mức 1: `get(x, 0)`. Mức 2: trả delta. Mức 3: dùng lại cho Bài 5.

**Bài 4.** Mức 1: tính `ans`. Mức 2: `m = max(ans)`. Mức 3: `ans.index(m)`.

**Bài 5.** Mức 1: cập nhật theo chuyển trạng thái đếm. Mức 2: thêm: `0→1` (+1), `1→2` (−1). Mức 3: bỏ: `1→0` (−1), `2→1` (+1); đối chiếu `sum(1 for v in Counter(w).values() if v == 1)`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2, 3
b1 = [Solution().distinctNumbers(n, k) for n, k in [([1, 2, 3, 4], 2), ([4, 4, 4], 1), ([1, 2, 1, 3, 1], 3), ([2, 2, 3, 3, 2, 2], 4)]]
print("Bài 1:", b1); assert b1 == [[2, 2, 2], [1, 1, 1], [2, 3, 2], [2, 2, 2]]
print("Bài 2:"); assert trace_array([5, 3, 5, 5, 3, 1], 3) == [2, 2, 2, 3]
def add_todo(cnt, x):
    delta = 1 if cnt.get(x, 0) == 0 else 0
    cnt[x] = cnt.get(x, 0) + 1; return delta
def remove_todo(cnt, x):
    cnt[x] -= 1
    if cnt[x] == 0:
        del cnt[x]; return -1
    return 0
c = {}; assert [add_todo(c, 2), add_todo(c, 2), remove_todo(c, 2), remove_todo(c, 2)] == [1, 0, 0, -1] and c == {}
print("Bài 3: OK")

Bài 1: [[2, 2, 2], [1, 1, 1], [2, 3, 2], [2, 2, 2]]
Bài 2:
i=0: thêm 5 (mới, +1)                 cửa sổ=[5]
i=1: thêm 3 (mới, +1)                 cửa sổ=[5, 3]
i=2: thêm 5 (đã có)                   cửa sổ=[5, 3, 5] -> ans[0] = 2
i=3: thêm 5 (đã có), bỏ 5 (còn 2)     cửa sổ=[3, 5, 5] -> ans[1] = 2
i=4: thêm 3 (đã có), bỏ 3 (còn 1)     cửa sổ=[5, 5, 3] -> ans[2] = 2
i=5: thêm 1 (mới, +1), bỏ 5 (còn 1)   cửa sổ=[5, 3, 1] -> ans[3] = 3
Bài 3: OK


In [28]:
# Bài 4, 5
def max_distinct_window(nums, k):
    ans = Solution().distinctNumbers(nums, k); m = max(ans)
    return m, ans.index(m)

def unique_once(nums, k):
    cnt = defaultdict(int); once = 0; res = []
    for i, x in enumerate(nums):
        cnt[x] += 1
        if cnt[x] == 1: once += 1
        elif cnt[x] == 2: once -= 1
        if i >= k:
            y = nums[i - k]; cnt[y] -= 1
            if cnt[y] == 0: once -= 1
            elif cnt[y] == 1: once += 1
        if i >= k - 1: res.append(once)
    return res

rng = random.Random(6)
for _ in range(400):
    n = rng.randint(1, 30); nums = [rng.randint(1, 6) for _ in range(n)]; k = rng.randint(1, n)
    m, idx = max_distinct_window(nums, k)
    assert m == max(len(set(nums[i:i + k])) for i in range(n - k + 1)) and len(set(nums[idx:idx + k])) == m
    assert unique_once(nums, k) == [sum(1 for v in Counter(nums[i:i + k]).values() if v == 1) for i in range(n - k + 1)]
print("Bài 4, 5: OK |", max_distinct_window([1, 2, 3, 2, 2, 1, 3], 3), unique_once([1, 2, 3, 2, 2, 1, 3], 3))

Bài 4, 5: OK | (3, 0) [3, 1, 1, 1, 3]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "cửa sổ cố định + số phần tử khác nhau":**

```python
cnt, distinct, ans = defaultdict(int), 0, []
for i, x in enumerate(nums):
    if cnt[x] == 0: distinct += 1
    cnt[x] += 1
    if i >= k:
        y = nums[i - k]; cnt[y] -= 1
        if cnt[y] == 0: distinct -= 1
    if i >= k - 1: ans.append(distinct)
return ans
```

**Checklist:**
- [ ] Đúng `n − k + 1` cửa sổ?
- [ ] Bỏ phần tử `nums[i − k]` (vừa rời cửa sổ), ghi đáp án khi `i ≥ k − 1`?
- [ ] `Counter` thì xóa khóa về 0; hoặc đếm `distinct` riêng?
- [ ] Kiểm tra "mới xuất hiện" **trước** khi tăng?
- [ ] Đã test `k = 1`, `k = n`, mọi phần tử bằng nhau, giá trị `10⁵`?